In [1]:
import requests


KOSIS_API_KEY = "ZTFhMjg1MzhmNmFiYWJlYmY3ZWUxZDA0ZDI2ZTM0YWU="

url = "https://kosis.kr/openapi/statisticsData.do"
params = {
    "method": "getList",
    "apiKey": KOSIS_API_KEY,
    "format": "json",
    "jsonVD": "Y",
    "orgId": "376",
    "tblId": "DT_376_90_SDMP042V_1",
    "startPrdDe": "201001",
    "endPrdDe": "202412"
}

r = requests.get(url, params=params)
data = r.json()


In [2]:
data

{'err': '20', 'errMsg': '필수요청변수값이 누락되었습니다.'}

In [3]:
import requests
from collections import defaultdict
from typing import Dict, List, Tuple, Any, Optional

KOSIS_PARAM_URL = "https://kosis.kr/openapi/Param/statisticsParameterData.do"

def _get_json(url: str, params: dict, timeout: int = 30) -> Any:
    r = requests.get(url, params=params, timeout=timeout)
    r.raise_for_status()
    return r.json()

def _safe_str(x) -> str:
    return "" if x is None else str(x)

def fetch_parameter_rows(api_key: str, org_id: str, tbl_id: str,
                         prd_se: str,
                         itm_id: str = "ALL",
                         obj_levels: Optional[Dict[str, str]] = None,
                         start_prd_de: Optional[str] = None,
                         end_prd_de: Optional[str] = None,
                         json_vd: str = "Y",
                         timeout: int = 30) -> List[dict]:
    """
    KOSIS 통계표 선택 방식(statisticsParameterData) 호출.
    obj_levels: {"objL1": "ALL" or code, "objL2": "...", ...}
    """
    params = {
        "method": "getList",
        "apiKey": api_key,
        "format": "json",
        "jsonVD": json_vd,
        "orgId": org_id,
        "tblId": tbl_id,
        "prdSe": prd_se,
        "itmId": itm_id,
    }
    # objL1은 필수. 없으면 기본 ALL 넣음
    obj_levels = obj_levels or {}
    if "objL1" not in obj_levels:
        obj_levels["objL1"] = "ALL"
    params.update(obj_levels)

    if start_prd_de:
        params["startPrdDe"] = start_prd_de
    if end_prd_de:
        params["endPrdDe"] = end_prd_de

    data = _get_json(KOSIS_PARAM_URL, params=params, timeout=timeout)

    # 에러 형태 처리
    if isinstance(data, dict) and "err" in data:
        raise RuntimeError(f"KOSIS error {data.get('err')}: {data.get('errMsg')} / params={params}")

    if not isinstance(data, list):
        raise RuntimeError(f"Unexpected response type: {type(data)} / head={str(data)[:300]}")

    return data

def parse_structure_from_rows(rows: List[dict]) -> Tuple[Dict[str, str], Dict[int, Dict[str, str]]]:
    """
    rows에서 itmId/itmName + C1~C8(code)/C1_NM~C8_NM(name) 추출
    returns:
      items: {itm_id: itm_nm}
      obj_levels: {level: {code: name}}  (level=1..8)
    """
    items = {}
    obj_levels = {i: {} for i in range(1, 9)}

    for r in rows:
        itm_id = _safe_str(r.get("ITM_ID"))
        itm_nm = _safe_str(r.get("ITM_NM"))
        if itm_id:
            items.setdefault(itm_id, itm_nm)

        for lv in range(1, 9):
            c = _safe_str(r.get(f"C{lv}"))
            n = _safe_str(r.get(f"C{lv}_NM"))
            if c:
                obj_levels[lv].setdefault(c, n)

    # 빈 레벨 제거
    obj_levels = {lv: mp for lv, mp in obj_levels.items() if mp}
    return items, obj_levels

def discover_kosis_structure(api_key: str, org_id: str, tbl_id: str,
                             prd_candidates: List[str] = ["M", "Q", "Y"],
                             max_depth: int = 3,
                             timeout: int = 30) -> dict:
    """
    - prdSe 후보(M/Q/Y)를 돌려서 응답이 되는 주기를 찾고
    - objL1..objL8 구조를 재귀적으로 최대 max_depth까지 탐색
    결과:
      {
        "prdSe_used": "...",
        "items": {itm_id: itm_nm, ...},
        "obj_levels": {1:{code:name}, 2:{...}, ...},
        "combos": [ {"prdSe":..., "itmId":..., "objL1":..., "objL2":...}, ... ]  # 샘플 조합
      }
    """
    # 1) prdSe 유효한 것 찾기
    prd_ok = None
    base_rows = None
    last_err = None

    for prd in prd_candidates:
        try:
            rows = fetch_parameter_rows(
                api_key, org_id, tbl_id,
                prd_se=prd,
                itm_id="ALL",
                obj_levels={"objL1": "ALL"},
                timeout=timeout
            )
            if rows:
                prd_ok = prd
                base_rows = rows
                break
        except Exception as e:
            last_err = e

    if prd_ok is None:
        raise RuntimeError(f"prdSe candidates all failed. last_err={last_err}")

    # 2) base에서 1차 파싱
    items, obj_levels = parse_structure_from_rows(base_rows)

    # 3) 재귀적으로 더 깊은 objL2~ 탐색 (C2가 실제로 존재하는지 확인하면서)
    combos = []

    def _recurse(level: int, current_obj: Dict[str, str]):
        """
        level: 다음으로 고정해볼 objL{level}
        current_obj: 이미 고정된 {"objL1": "...", "objL2":"...", ...}
        """
        # 샘플 조합 저장 (itmId는 일단 대표 1개만)
        rep_itm = next(iter(items.keys())) if items else "ALL"
        combos.append({"prdSe": prd_ok, "itmId": rep_itm, **current_obj})

        if level > max_depth:
            return

        # 다음 레벨 후보 코드를 얻기 위해: itmId=ALL로 호출해서 C{level}이 나오면 그걸 수집
        try:
            rows = fetch_parameter_rows(
                api_key, org_id, tbl_id,
                prd_se=prd_ok,
                itm_id="ALL",
                obj_levels=current_obj,
                timeout=timeout
            )
        except Exception:
            return

        _, deeper_obj = parse_structure_from_rows(rows)
        # deeper_obj에서 level에 해당하는 C{level}이 없으면 중단
        if level not in deeper_obj:
            return

        # 전역 obj_levels에 merge
        for lv, mp in deeper_obj.items():
            obj_levels.setdefault(lv, {})
            for k, v in mp.items():
                obj_levels[lv].setdefault(k, v)

        # 각 후보 코드로 더 내려가기
        next_codes = list(deeper_obj[level].keys())
        for code in next_codes:
            nxt = dict(current_obj)
            nxt[f"objL{level}"] = code
            _recurse(level + 1, nxt)

    # objL1부터 시작: objL1 후보 코드가 있으면 각각 내려가며 탐색
    if 1 in obj_levels and obj_levels[1]:
        for c1 in list(obj_levels[1].keys()):
            _recurse(2, {"objL1": c1})
    else:
        # objL1 후보가 안 잡히면 ALL로만
        _recurse(2, {"objL1": "ALL"})

    return {
        "prdSe_used": prd_ok,
        "items": items,
        "obj_levels": obj_levels,
        "combos": combos[:200],  # 너무 길어질 수 있으니 샘플만
        "base_row_count": len(base_rows),
    }

def pretty_print_structure(result: dict, top_n: int = 50):
    print(f"\n✅ prdSe_used = {result['prdSe_used']}")
    print(f"✅ base_row_count = {result['base_row_count']}\n")

    items = result["items"]
    print(f"[ITM] items count = {len(items)}")
    for i, (k, v) in enumerate(items.items()):
        if i >= top_n:
            print(f"... ({len(items)-top_n} more)")
            break
        print(f"  - {k}: {v}")

    obj_levels = result["obj_levels"]
    print("\n[OBJ] classification levels discovered:")
    for lv in sorted(obj_levels.keys()):
        mp = obj_levels[lv]
        print(f"  - C{lv} count = {len(mp)}")
        for j, (k, v) in enumerate(mp.items()):
            if j >= min(top_n, 30):
                print(f"    ... ({len(mp)-min(top_n,30)} more)")
                break
            print(f"    * {k}: {v}")

    print("\n[COMBOS] sample callable parameter combos (first 10):")
    for c in result["combos"][:10]:
        print("  ", c)

# -------------------------
# 실행 예시
# -------------------------
# result = discover_kosis_structure(
#     api_key=KOSIS_API_KEY,
#     org_id="376",
#     tbl_id="DT_376_90_SDMP042V_1",
#     prd_candidates=["M", "Q", "Y"],
#     max_depth=3
# )
# pretty_print_structure(result)


In [4]:
result = discover_kosis_structure(
    api_key=KOSIS_API_KEY,
    org_id="376",
    tbl_id="DT_376_90_SDMP042V_1",
    prd_candidates=["M", "Q", "Y"],
    max_depth=3
)
pretty_print_structure(result)

RuntimeError: prdSe candidates all failed. last_err=KOSIS error 20: 필수요청변수값이 누락되었습니다. (objL) / params={'method': 'getList', 'apiKey': 'ZTFhMjg1MzhmNmFiYWJlYmY3ZWUxZDA0ZDI2ZTM0YWU=', 'format': 'json', 'jsonVD': 'Y', 'orgId': '376', 'tblId': 'DT_376_90_SDMP042V_1', 'prdSe': 'Y', 'itmId': 'ALL', 'objL1': 'ALL'}

In [5]:
result

NameError: name 'result' is not defined